# 01 — Herschel–Bulkley Transient Solver
Self‑contained solver for 1D single‑phase, weakly compressible Herschel–Bulkley flow in a circular pipe (semi‑implicit, staggered grid).

In [ ]:

"""
hb_transient.py
---------------
1D transient single-phase Herschel–Bulkley fluid in a circular pipe (horizontal).
- Staggered grid: pressure at cells, velocity at faces.
- Weak compressibility via bulk modulus B (rho = rho0 * (1 + (p - p0)/B)).
- Semi-implicit: explicit convection, implicit pressure correction, friction linearized.
- Inlet: prescribed volumetric flow rate Q_in(t).
- Outlet: prescribed pressure p_out (Dirichlet).
Author: ChatGPT
"""

from dataclasses import dataclass
import numpy as np

@dataclass
class HBParams:
    tau_y: float  # yield stress [Pa]
    K: float      # consistency [Pa*s^n]
    n: float      # flow index [-]

@dataclass
class PipeParams:
    L: float      # length [m]
    D: float      # diameter [m]
    A: float      # area [m^2]
    R: float      # radius [m]

@dataclass
class FluidCompress:
    rho0: float   # reference density [kg/m^3]
    p0: float     # reference pressure [Pa]
    B: float      # bulk modulus [Pa]

@dataclass
class Numerics:
    dx: float     # cell size [m]
    dt: float     # time step [s]
    cfl: float    # target CFL for guidance (not enforced strictly)

def build_pipe(L, D):
    A = 0.25*np.pi*D**2
    return PipeParams(L=L, D=D, A=A, R=0.5*D)

# --- Herschel–Bulkley helpers ---

def hb_Q_from_dpdx(dpdx, D, tau_y, K, n, nr=300):
    """
    Volumetric flow rate Q [m^3/s] in laminar, fully-developed pipe flow for HB fluid
    at given axial pressure gradient dpdx [Pa/m].
    """
    R = D/2.0
    tau_w = (D/4.0) * dpdx
    if tau_w <= tau_y:  # plug, no flow
        return 0.0

    # radial grid (center to wall)
    r = np.linspace(0.0, R, nr)
    tau = tau_w * (r/R)
    gdot = np.zeros_like(r)
    mask = tau > tau_y
    gdot[mask] = ((tau[mask] - tau_y) / K)**(1.0/n)

    # velocity profile: v(r) = \int_r^R gdot(s) ds  (no slip at wall)
    # integrate from center outward using cumulative trapezoid
    dr = r[1]-r[0]
    # integrate gdot from wall toward center
    # compute integral G(r) = integral from r to R of gdot(s) ds
    # approximate by reversing arrays
    g_rev = gdot[::-1]
    G_rev = np.cumsum(g_rev) * dr
    G = G_rev[::-1]
    v = G  # v(R)=0 at wall

    # Q = 2π ∫ v(r) r dr
    Q = 2*np.pi * np.trapz(v * r, r)
    return Q

def hb_tau_w_from_u(u, D, tau_y, K, n, tol=1e-10, maxit=60):
    """
    Given mean axial velocity u [m/s], invert the steady HB pipe relation to find
    the wall shear stress tau_w [Pa] that would produce this flow in laminar regime.
    Uses scalar root find on dpdx via hb_Q_from_dpdx and then tau_w = D/4 dpdx.
    """
    A = 0.25*np.pi*D**2
    Q_target = abs(u) * A
    if Q_target == 0:
        return 0.0
    # bracket dpdx
    dp_lo = 1e-6
    Q_lo = hb_Q_from_dpdx(dp_lo, D, tau_y, K, n)
    dp_hi = dp_lo
    Q_hi = Q_lo
    it = 0
    while Q_hi < Q_target and it < 60:
        dp_hi *= 2.0
        Q_hi = hb_Q_from_dpdx(dp_hi, D, tau_y, K, n)
        it += 1
    # bisection
    for _ in range(maxit):
        mid = 0.5*(dp_lo + dp_hi)
        Q_mid = hb_Q_from_dpdx(mid, D, tau_y, K, n)
        if abs(Q_mid - Q_target) <= tol * max(Q_target, 1.0):
            dp_star = mid
            break
        if Q_mid < Q_target:
            dp_lo = mid
        else:
            dp_hi = mid
    else:
        dp_star = 0.5*(dp_lo + dp_hi)

    tau_w = (D/4.0) * dp_star
    return tau_w

def hb_tau_w_and_dtau_du(u, D, tau_y, K, n):
    """
    Return tau_w(u) and numerical derivative d(tau_w)/du for linearization.
    """
    tau0 = hb_tau_w_from_u(u, D, tau_y, K, n)
    # numerical slope via small relative perturbation
    du = max(1e-6, 1e-3*max(1.0, abs(u)))
    tau1 = hb_tau_w_from_u(u + du, D, tau_y, K, n)
    dtau = (tau1 - tau0) / du
    return tau0, dtau

# --- Solver core ---

class HBTransient1D:
    def __init__(self, pipe: PipeParams, fluid: FluidCompress, hb: HBParams,
                 numerics: Numerics, p_out: float):
        self.pipe = pipe
        self.fluid = fluid
        self.hb = hb
        self.num = numerics
        # Grid
        self.N = int(round(pipe.L / numerics.dx))
        self.dx = pipe.L / self.N
        self.xc = (np.arange(self.N) + 0.5) * self.dx  # cell centers
        self.xf = np.arange(self.N+1) * self.dx        # faces
        # Unknowns
        self.p = np.ones(self.N) * fluid.p0            # cell pressure
        self.u = np.zeros(self.N+1)                    # face velocity
        # Outlet pressure (Dirichlet at right boundary)
        self.p_out = p_out

    def density(self, p):
        f = self.fluid
        return f.rho0 * (1.0 + (p - f.p0)/f.B)

    def step(self, t, Q_in_fn):
        """
        Advance one time step using semi-implicit scheme.
        Q_in_fn : function giving inlet volumetric flow rate Q(t) [m^3/s].
        """
        D = self.pipe.D
        A = self.pipe.A
        rho_c = self.density(self.p)            # cell-centered rho^n
        rho_f = 0.5 * (np.r_[rho_c[0], rho_c] + np.r_[rho_c, rho_c[-1]])  # simple average to faces

        dt = self.num.dt
        dx = self.dx
        N = self.N

        # --- 1) Momentum: build u* with friction linearized implicitly; pressure uses p^n
        u_old = self.u.copy()
        u_star = self.u.copy()

        # pressure gradient at faces using p^n and outlet Dirichlet
        p_ext = np.r_[self.p, self.p_out]  # size N+1 for right faces
        dpdx_f = np.zeros(N+1)
        # internal faces 1..N-1 use jump of neighboring cells
        dpdx_f[1:-1] = (self.p[1:] - self.p[:-1]) / dx
        # left boundary uses ghost such that pressure gradient aligns with first cell (Neumann from momentum)
        # but we will override velocity with inlet BC anyway after pressure correction
        dpdx_f[0] = (self.p[0] - self.p[0]) / dx
        # right boundary: last face between cell N-1 and outlet pressure
        dpdx_f[-1] = (self.p_out - self.p[-1]) / dx

        # Friction linearization at faces: tau_w ≈ a + b * u
        a_f = np.zeros(N+1)
        b_f = np.zeros(N+1)
        for fidx in range(N+1):
            tau0, dtau = hb_tau_w_and_dtau_du(u_old[fidx], D, self.hb.tau_y, self.hb.K, self.hb.n)
            # Use sign of u in tau (direction), approximate linear model around u_old
            a_f[fidx] = tau0 - dtau * u_old[fidx]
            b_f[fidx] = dtau  # so tau ≈ a + b*u

        # Build u* from: (rho/Δt + (4/D) b) u* = RHS = rho u^n/Δt - dpdx + gravity(0) - (4/D) a
        # gravity neglected in this horizontal model
        lhs = (rho_f/dt) + (4.0/D) * b_f
        rhs = (rho_f/dt) * u_old - dpdx_f - (4.0/D) * a_f

        u_star = rhs / np.maximum(lhs, 1e-20)

        # Apply provisional inlet velocity from Q_in (will be corrected again by pressure solve)
        u_star[0] = Q_in_fn(t + dt) / A

        # --- 2) Pressure solve (linear): enforce continuity with compressibility
        # u^{n+1} = u* - (dt / rho_f) * dpdx^{n+1}
        # mass balance in cell i:
        # (rho^{n+1} - rho^n)/dt + (1/A) * (rho_f u^{n+1})_{i+1/2} - (rho_f u^{n+1})_{i-1/2} / dx = 0
        # rho^{n+1} = rho^n + rho0/B * (p^{n+1} - p^n)

        rho0 = self.fluid.rho0
        B = self.fluid.B
        # Coefficients for tridiagonal A p^{n+1} = b
        a = np.zeros(N)  # lower diag
        b = np.zeros(N)  # main diag
        c = np.zeros(N)  # upper diag
        rhs_p = np.zeros(N)

        # Precompute coefficients for face velocity dependence on pressure
        # dpdx^{n+1} at face f between cells i (left) and i+1 (right) is (p_{i+1}^{n+1} - p_i^{n+1})/dx
        # u^{n+1}_f = u*_f - (dt / rho_f) * (p_R - p_L)/dx
        alpha_f = dt / (rho_f * dx)  # coefficient multiplying (p_R - p_L)

        # Flux at face f into cell i is F_f = rho_f * u^{n+1}_f * A
        # We linearize density at faces using average of cell densities (which depend on p^{n+1})
        # rho_c^{n+1} = rho_c^n + (rho0/B) (p^{n+1} - p^n)
        rho_c_n = rho_c.copy()

        # helper to add contributions
        for i in range(N):
            # left face index fL=i, right face fR=i+1
            fL = i
            fR = i+1
            # Stencil for p at faces:
            # u^{n+1}_{fL} = u*_{fL} - alpha_{fL} * (p_i^{n+1} - p_{i-1}^{n+1}); (for i=0, p_{-1} is ghost -> use inlet Neumann (0) )
            # u^{n+1}_{fR} = u*_{fR} - alpha_{fR} * (p_{i+1}^{n+1} - p_i^{n+1}); (for i=N-1, p_{i+1} = p_out is known Dirichlet)

            # Face densities: average of neighbor cell densities, which are linear in p^{n+1}
            # Left face neighbors: i-1 and i (use i for boundary)
            if i == 0:
                rhoL_L = rho_c_n[i]   # neighbor "i-1" -> use i
                rhoL_R = rho_c_n[i]
            else:
                rhoL_L = rho_c_n[i-1]
                rhoL_R = rho_c_n[i]

            if i == N-1:
                rhoR_L = rho_c_n[i]
                rhoR_R = rho_c_n[i]   # neighbor "i+1" -> use i
            else:
                rhoR_L = rho_c_n[i]
                rhoR_R = rho_c_n[i+1]

            # Coefficients for rho at faces w.r.t cell pressures
            # rho_face = 0.5*(rhoL + rhoR) where rho = rho_n + (rho0/B)(p^{n+1}-p^n)
            # => dependence adds (rho0/(2B)) on the two neighboring p's.
            rcoef = rho0/(2.0*B)

            # Build contributions for mass balance:
            # (rho^{n+1}_i - rho^n_i)/dt  term:
            b[i] += (rho0/B) / dt

            # Flux through left face: F_L = A * rho_fL * u^{n+1}_{fL}
            # Contribution from pressure via u^{n+1}:
            # u^{n+1}_{fL} has p_i and p_{i-1}. For i=0 we drop p_{-1}.
            if i > 0:
                # coefficient for p_{i-1}
                a[i] +=  (self.pipe.A) * (+alpha_f[fL]) / dx  * 0  # placeholder to keep structure
                # Actually, alpha already had 1/dx, we used it as dt/(rho*dx); keep as is
                pass

            # We will assemble via explicit formulas below to avoid mistakes.

        # Re-assemble carefully in a second pass
        a[:] = 0.0; b[:] = (rho0/B)/dt; c[:] = 0.0; rhs_p[:] = 0.0

        # Precompute linearized face density coefficients: for each face f, identify left and right cell indices
        face_left = np.arange(N+1) - 1
        face_right = np.arange(N+1)
        face_left[0] = 0
        face_right[0] = 0
        face_left[-1] = N-1
        face_right[-1] = N-1

        # Precompute dependence of face mass flux on cell pressures:
        # m_f = A * rho_f * u^{n+1}_f
        # rho_f = 0.5*(rho_i + rho_j) = 0.5*(rho_i^n + rho_j^n) + rcoef*(p_i^{n+1}-p_i^n + p_j^{n+1}-p_j^n)
        # u^{n+1}_f = u*_f - alpha_f*(p_j^{n+1}-p_i^{n+1})
        # Expand and collect coefficients for p_i and p_j into the two neighboring mass balances.
        rcoef = self.fluid.rho0/(2.0*self.fluid.B)
        Asec = self.pipe.A

        # We'll build the discrete divergence operator contributions:
        for i in range(N):
            # LEFT FACE
            fL = i
            iL = face_left[fL]
            iR = face_right[fL]
            # Known terms (independent of p^{n+1}):
            rho_f_n = 0.5*(rho_c_n[iL] + rho_c_n[iR])
            m_star_L = Asec * rho_f_n * u_star[fL]
            # Pressure coefficients:
            # From velocity part: -A*rho_f_n * alpha_f * (p_R - p_L)
            coef_L_from_u_pL = +Asec * rho_f_n * alpha_f[fL]
            coef_L_from_u_pR = -Asec * rho_f_n * alpha_f[fL]
            # From density part multiplying u*:
            coef_L_from_rho_pL = Asec * rcoef * u_star[fL]
            coef_L_from_rho_pR = Asec * rcoef * u_star[fL]

            # RIGHT FACE
            fR = i+1
            iL2 = face_left[fR]
            iR2 = face_right[fR]
            rho_f_n_R = 0.5*(rho_c_n[iL2] + rho_c_n[iR2])
            m_star_R = Asec * rho_f_n_R * u_star[fR]
            coef_R_from_u_pL = +Asec * rho_f_n_R * alpha_f[fR]
            coef_R_from_u_pR = -Asec * rho_f_n_R * alpha_f[fR]
            coef_R_from_rho_pL = Asec * rcoef * u_star[fR]
            coef_R_from_rho_pR = Asec * rcoef * u_star[fR]

            # Divergence term: (m_R - m_L)/dx
            # Contributions to equation for cell i:
            # p_i coefficient gets:
            #  - from left face: (coef_L_from_u_pR for p_i if i is right of fL) or (coef_L_from_u_pL if i is left) etc.
            # Here, for left face, (iR == i) and for right face, (iL2 == i)
            b[i] += (coef_R_from_u_pL if iL2 == i else coef_R_from_u_pR) / dx
            b[i] -= (coef_L_from_u_pR if iR == i else coef_L_from_u_pL) / dx

            # neighbor i-1
            if i > 0:
                a[i] += (coef_L_from_u_pL if iL == i-1 else coef_L_from_u_pR) / dx
                # density part couples too:
                a[i] += (coef_L_from_rho_pL if iL == i-1 else 0.0) / dx
            # neighbor i+1
            if i < N-1:
                c[i] += (coef_R_from_u_pR if iR2 == i+1 else coef_R_from_u_pL) / dx
                c[i] += (coef_R_from_rho_pR if iR2 == i+1 else 0.0) / dx

            # self contributions from density parts
            b[i] += (coef_L_from_rho_pR if iR == i else coef_L_from_rho_pL) / dx
            b[i] -= (coef_R_from_rho_pL if iL2 == i else coef_R_from_rho_pR) / dx

            # RHS contains explicit parts and terms with p^n
            rhs_p[i] = -(m_star_R - m_star_L)/dx + (rho_c_n[i] - rho_c_n[i])/dt  # second term cancels

            # Add terms related to known p^n in density linearization: -A*rcoef*u* * ( -p^n contributions cancel in both faces )
            # Since we wrote rho^{n+1} = rho^n + rcoef*(p^{n+1}-p^n)*2 (face), the -p^n parts cancel between L and R in divergence.
            # So no extra rhs here beyond m* divergence.

        # Boundary conditions for pressure system:
        # - Left boundary: no pressure Dirichlet; inlet is flow. So natural (Neumann) is embedded via flux form.
        # - Right boundary: p_out Dirichlet. We enforce by modifying last equation using ghost: for face N, u^{n+1}_N uses p_out.
        #   Our coefficients already treat p_out via dpdx in u*, but the linear system still needs a fix: add constant from velocity correction.
        # Implement p_out by shifting rhs: for cell N-1, the right-face velocity correction term includes p_out
        # u^{n+1}_N = u*_N - alpha_N * (p_out - p_{N-1}^{n+1})
        # The mass flux through right face contains -A*rho_f_n * alpha_N * p_out  (moves to rhs)
        # and density part also has rcoef * A * u*_N * p_out (moves to rhs).

        # Apply right boundary p_out to rhs through the divergence linearization
        i = N-1
        fR = N
        iL2 = face_left[fR]
        iR2 = face_right[fR]
        rho_f_n_R = 0.5*(rho_c_n[iL2] + rho_c_n[iR2])
        # Velocity part contribution with p_out:
        rhs_p[i] += (self.pipe.A * rho_f_n_R * alpha_f[fR] * self.p_out) / dx
        # Density part with p_out:
        rhs_p[i] -= (self.pipe.A * rcoef * u_star[fR] * self.p_out) / dx

        # Solve tridiagonal
        p_new = solve_tridiagonal(a, b, c, rhs_p)

        # --- 3) Correct velocity with new pressure
        pL = np.r_[p_new[0], p_new]
        pR = np.r_[p_new, self.p_out]
        self.u = u_star - (dt / rho_f) * (pR - pL) / dx

        # Enforce inlet flow exactly: overwrite left face velocity
        self.u[0] = Q_in_fn(t + dt) / A

        # Update pressure
        self.p = p_new

        # Optional: mild end corrections to avoid drift at outlet
        return

def solve_tridiagonal(a, b, c, d):
    """
    Solve tridiagonal system with vectors a (subdiag), b (diag), c (superdiag), d (rhs).
    Thomas algorithm.
    """
    n = len(b)
    ac, bc, cc, dc = map(np.array, (a.copy(), b.copy(), c.copy(), d.copy()))
    for i in range(1, n):
        mc = ac[i] / bc[i-1] if bc[i-1] != 0 else 0.0
        bc[i] = bc[i] - mc * cc[i-1]
        dc[i] = dc[i] - mc * dc[i-1]
    xc = np.zeros(n)
    xc[-1] = dc[-1] / bc[-1]
    for i in range(n-2, -1, -1):
        xc[i] = (dc[i] - cc[i] * xc[i+1]) / bc[i]
    return xc
